# <center> **RL Эксперименты с кастомной средой**

## <center> **1. Введение**

В рамках проекта рассматривается задача обучения агента с подкреплением в визуальной игровой среде **HeliRescue**, созданной на основе 2D‑игры на Pygame с управляемым вертолетом, падающим парашютистом и вражеским объектом alien. Агенту предоставляется доступ только к пиксельному представлению текущего кадра, а управление осуществляется через дискретный набор действий, имитирующих нажатия клавиш направления в исходной игре.

<img src="ScreenShots/screen1.png" alt="Game ScreenShot" style="width: 60%; height: 30%;">

**Основная цель проекта** — спроектировать и реализовать кастомную среду в формате Gymnasium, совместимую со Stable-Baselines3 и политиками типа CnnPolicy, провести серию экспериментов по обучению агента и проанализировать получающиеся стратегии поведения. Такой выбор постановки позволяет исследовать особенности визуального RL: влияние разрешения наблюдений, частоты действий (frame_skip) и формы функции награды на качество выученной политики в среде с непрерывным пространством состояний и дискретным управлением.

## <center> **2. Описание среды**

### **Пространство состояний**

Пространство состояний непрерывно и задается в виде $RGB$-изображения игрового кадра, сформированного с помощью pygame. На каждом шаге агент получает наблюдение $s_{t}$ в виде массива $uint8$ размера $(H, W, 3)$, где $H$ и $W$ - высота и ширина кадра после масштабирования масштабирования $84 X 84$ для использования в стандартной CnnPolicy в Stable-Baselines3. На изображении присутсивуют фон, вертолет, парашютист и пришелец, что делает задачу полностью наблюдаемой, но в визуальном, а не табличном виде.

Таким образом, состояние среды включает положения и скорости объектов, но агент не получает их в явном виде, а вынужден извлекать необходимые признаки (позиции, относительную геометрию, направление движения) напрямую из изображения с помощью сверточной нейронной сети.

### **Пространство действий**

Пространство действий дискретно и определяется как множество из пяти элементарных команд, соответствующих возможным перемещениям вертолета:

+ $0$ — отсутствие действия (NOOP, вертолет остается на месте по горизонтали и вертикали).
+ $1$ — движение влево при условии, что вертолет не выходит за левую границу игрового поля.
+ $2$ — движение вправо при условии, что вертолет не выходит за правую границу.
+ $3$ — движение вверх до верхней границы игрового окна.
+ $4$ — движение вниз до нижней границы игрового окна.

Наличие NOOP позволяет агенту при необходимости стабилизировать положение и не совершать лишних движений.

### **Функция награды**

Функция награды построена по мотивам игровой логики и поощряет стратегию действий, приводящую к спасению парашютиста и избеганию опасных ситуаций.

Столкновения:

+ Вертолет - парашютист: $+1$
+ Пришелец - парашютист: $-1$
+ Вертолет - пришелец: $-10$ и завершение эпизода.

Функция награды задается как скалярная величина:

$r_{t} = R(s_{t}, a_{t}, s_{t+1})$,

то есть как значение, зависящее от текущего состояния среды, действия агента и следующего состояния. В рассматриваемой задаче награда отражает основные действия игры.

$r_{t} = 1_{save}(t) - 1_{lose}(t) - 10 \cdot 1_{crash}(t) + 10 \cdot 1_{win}(t)$, где:

+ $1_{save}(t)$ - спасение парашютиста на шаге $t$
+ $1_{lose}(t)$ - перехват парашютиста пришельцем
+ $1_{crash}(t)$ - столкновение вертолета с пришельцем
+ $1_{win}(t)$ - достижение победы

### **Критерии успеха**

Критерием успеха в одном эпизоде является достижение целевого значения счета $+10$ до того, как счет опустится до $−10$ или будет превышен лимит по числу шагов эпизода. Эпизодическое завершение по достижению порога $+10$ интерпретируется как "победа" агента, так как накопленная серия успешных спасений перевешивает возможные ошибки и отражает устойчивую стратегию контроля вертолета.

In [2]:
import os 
import numpy as np  
import pygame as pg 
import gymnasium as gym
import warnings
import matplotlib.pyplot as plt
from pathlib import Path 
from random import Random 
from typing import Any 
from gymnasium import spaces

warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
# Класс среды, совместимый с gymnasium
class HeliRescueEnv(gym.Env):
    metadata = {
        'render_modes': ['human', 'rgb_array'],
        'render_fps': 60
    }
    
    # Действия агента
    ACTION_NOOP = 0
    ACTION_LEFT = 1
    ACTION_RIGHT = 2
    ACTION_UP = 3
    ACTION_DOWN = 4 
    
    def __init(
        self,
        assets_dir: str | os.PathLike | None = None,
        render_mode: str | None = None,
        obs_size: tuple[int, int] = (84, 84),
        window_size: tuple[int, int] = (1000, 700),
        stop_score: int = 10,
        max_steps: int = 2000,
        helicopter_speed: int = 10,
        skydiver_speed_range: tuple[int, int] = (2, 5),
        alien_speed_range: tuple[int, int] = (5, 10),
        frame_skip: int = 1,
        seed: int | None = None
        ):
        """
        Инициализирует среду HeliRescueEnv

        Args:
            assets_dir (str | os.PathLike | None, optional): путь к директории проекта, в которой находятся игровые ресурсы. Defaults to None.
            render_mode (str | None, optional): способ визуализации среды. Defaults to None.
            obs_size (tuple[int, int], optional): размер итогового наблюдения, которое поступает агенту на вход в виде изображения. Defaults to (84, 84).
            window_size (tuple[int, int], optional): реальный размер игровой сцены в пикселях. Defaults to (1000, 700).
            stop_score (int, optional): порог счета, при достижении которого эпизод завершается. Defaults to 10.
            max_steps (int, optional): задает максимальную длину одного эпизода в шагах среды. Defaults to 2000.
            helicopter_speed (int, optional): на сколько пикселей перемещается вертолет за один шаг. Defaults to 10.
            skydiver_speed_range (tuple[int, int], optional): диапазон случайной скорости падения парашютиста. Defaults to (2, 5).
            alien_speed_range (tuple[int, int], optional): задает диапазон случайной скорости движения пришельца. Defaults to (5, 10).
            frame_skip (int, optional): определяет, сколько внутренних игровых обновлений выполняется после одного действия агента. Defaults to 1.
            seed (int | None, optional): начальное значение генератора случайных чисел среды. Defaults to None.
        """
        
        self.render_mode = render_mode        
        self.obs_width, self.obs_height = obs_size       
        self.win_width, self.win_height = window_size        
        self.stop_score = stop_score       
        self.max_steps = max_steps        
        self.helicopter_speed = helicopter_speed        
        self.skydiver_speed_range = skydiver_speed_range        
        self.alien_speed_range = alien_speed_range        
        self.frame_skip = max(1, int(frame_skip))        
        self._rng = Random(seed) # генератор случайных чисел        
        self.action_space = spaces.Discrete(5) # Пространство действий: 5 дискретных действий
        # Пространство наблюдений: цветное изображение размера (H, W, 3).
        self.observation_space = spaces.Box(
            low=0,
            high=255,
            shape=(self.obs_height, self.obs_width, 3),
            dtype=np.uint8,
        )
        
        self.assets_dir = Path(assets_dir) if assets_dir is not None else Path('.').resolve() # Папка с ассетами       
        self.images_dir = self.assets_dir / 'images' # Папка с изображениями        
        self._pg_initialized = False # Флаг инициализации pygame        
        self._display = None        
        self._canvas = None # Внутренний холст, на котором рисуется кадр игры      
        self._clock = None
        self._font = None

        # Изображения игровых объектов Pygame Surface
        self.background = None
        self.helicopter_right_img = None
        self.helicopter_left_img = None
        self.skydiver_img = None
        self.alien_img = None

        # Pygame Rect
        self.helicopter_rect = None
        self.skydiver_rect = None
        self.alien_rect = None        
        self.helicopter_img = None # Текущее изображение вертолета зависит от направления
                
        self.score = 0 # Игровой счет        
        self.steps = 0 # Счетчик шагов текущего эпизода        
        self.win = False # Флаг победы        
        self._ensure_pygame() # Инициализируем pygame        
        self._load_assets() # Загружаем ассеты        
        self._build_game_objects() # Создаем игровые объекты
        
    def _ensure_pygame(self) -> None:
        """
            Инициализирует pygame 
        """
        if self._pg_initialized:
            return
        if not pg.get_init():
            pg.init()
        if not pg.font.get_init():
            pg.font.init()
            
        self._canvas = pg.Surface((self.win_width, self.win_height))
        self._clock = pg.time.Clock()
        self._font = pg.font.SysFont("arial", 28, bold=True)
        
        if self.render_mode == "human":
            self._display = pg.display.set_mode((self.win_width, self.win_height))
            pg.display.set_caption("HeliRescueEnv")
        self._pg_initialized = True